# Практика · Тематичне моделювання

> Лекція: [lecture.html](lecture.html) · Домашнє завдання: [homework.html](homework.html) ·
> Тест: [quiz.html](quiz.html)

Тут рахуються **всі** числа, які називає лекція, у тому самому порядку.

Що зробимо:

1. Зберемо корпус і побудуємо стоп-список прямо з його частот.
2. Запустимо LDA і NMF на сирому мішку слів — і побачимо, що обидві дають службові слова.
3. Придумаємо **число**, яким міряти читабельність тем, і перевіримо його на очевидних прикладах.
4. Пройдемо драбину прибирання: стоп-слова → `max_df` → лематизація → TF-IDF.
5. Розберемо NMF руками: перемножимо дві матриці й звіримо похибку з `sklearn`.
6. Заміряємо перплексію LDA й помилку відновлення NMF для чотирьох значень `k`.
7. Перевіримо, чи повторюються теми між прогонами й між вибірками.
8. Спитаємо, чи видно різницю LDA і NMF на справжній задачі.

> ⏱ Заміряно наодинці на чотирьох ядрах без відеокарти: близько **трьох хвилин**.
> На зайнятій машині виходить до пʼяти — час LDA дуже чутливий до навантаження.
>
> Найдовший крок — **LDA на 20 000 документів**, вона одна йде близько **пів
> хвилини**. Решта заміру навмисне врізана, інакше зошит не вклався б: драбина,
> стійкість і підвибірки рахуються на **1 200** документах, кількість тем — на
> **800**, задача класифікації — на **3 000**. Що саме врізано, сказано в кожному
> розділі.

## 1 · Середовище

Перша клітинка друкує версії. Якщо в тебе інші — числа можуть поїхати,
і краще знати про це одразу.

In [ ]:
# Потоки фіксуємо ДО імпорту numpy — інакше OpenMP розкидає роботу по ядрах,
# і на спільній машині процесорний час роздувається вдесятеро на порожньому місці.
import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'

import sys, re, glob, gettext, time, warnings
import numpy as np
import sklearn
import pymorphy3
from collections import Counter
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation, NMF
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from scipy.optimize import linear_sum_assignment

warnings.filterwarnings('ignore')          # sklearn охоче попереджає про збіжність

print("Python   ", sys.version.split()[0])
print("numpy    ", np.__version__)
print("sklearn  ", sklearn.__version__)
print("pymorphy3", pymorphy3.__version__)
print("ядер у системі:", os.cpu_count(), "· потоків дозволено: 1")

## 2 · Корпус

Той самий, що в усьому блоці: українські переклади інтерфейсів із `.mo`-файлів
системи. Це справжня українська мова, вона лежить на диску (жодної мережі) і не
міняється між запусками.

Чого від нього не варто чекати: це вузький домен — технічна лексика, короткі
речення, наказовий спосіб. Медіанний документ має **шість** слів, і саме це
вилізе боком у розділі про кількість тем.

⚠️ **Без української локалі цей зошит до кінця не виконається.** Нижче стоїть запобіжник, який вмикає вбудований мінікорпус, і перші розділи на ньому справді працюють. Але це сотня документів замість сотень тисяч, і далі розділам просто бракує матеріалу: відсів лишає `CountVectorizer` без ознак, а вибірки просять більше прикладів, ніж є. Зошит упаде на зрозумілій помилці, а не мовчки — це заміряно, а не припущено. Перевірити, чи корпус є: `ls /usr/share/locale/uk/LC_MESSAGES/*.mo | wc -l` — потрібно приблизно від сотні файлів.

In [ ]:
FALLBACK = [
"перекодувати метадані в це кодування",
"Перевищено час очікування на введення-виведення з гнізда",
"Буде проігноровано, зберігається для сумісності",
"Не вдалося отримати дані про ємність акумулятора",
"Додає маломасштабний растр, подібний до локальної зернистості",
"Налаштувати таблицю клавіш кани",
"Додати тло документа до кожного перетвореного шару.",
"Обмежити загальне використання чорнил на аркуш паперу",
"Вікно віджету, яке реалізовано",
"Мінімальне розширення дочірнього елемента",
"Вам слід включити принаймні один диск як ціль встановлення.",
"швейцарська німецька мова жестів",
"Виконано забагато спроб отримання пароля",
"Андаманські і Нікобарські острови",
"Чи слід підвкладкам заповнювати виділену ділянку",
"Увімкнути помилку подвійного подавання через товщину паперу",
"Однорідний за розміром вертикально",
"Некоректний формат прав доступу",
"Неочікуване завершення формального виразу",
"Більше старих транзакцій немає.",
"Некоректні параметри режиму знімання",
"Тритлумити яскравість екрана після періоду неактивності",
"Використовувати спекуляцію даних після перезавантаження.",
"вказана адреса основного сервера ключів є некоректною",
"не вдалося створити знімок вікна",
"Порогове значення порожніх сторінок для програмного відкидання",
"не перевіряти шляхи символічних посилань на файли",
"Збільшений час очікування лампи",
"не є коректною числовою специфікацією",
"пароль користувача у форматі звичайного тексту",
"не вдалося отримати повідомлення від батьківського процесу",
"Точне налаштування балансу білого",
"Ширина паперу, на якому слід надрукувати дані",
"показувати також ідентифікатори груп",
"УВАГА: цілісність повідомлення не захищено",
"Опале листя восени або щось вкрите живим листям",
"Під час приготування документа сталася помилка",
"помилка у додатку підтвердження",
"Відстань за вертикаллю між двома дочірніми віджетами",
"не можна комбінувати кілька визначників фільтра",
"Не вдалося оновити файл налаштування.",
"в заголовку файлу не знайдено магічного рядка",
"Динамічна область, найближчий предмет",
"несумісні типи в бінарному виразі",
"Не вдалося обробити відповідь сервера",
"Не вдалося побудувати назву інтерфейсу",
"Визначає, чи може бути встановлено часткову відповідність",
"вивести список усіх пакунків неактивних потоків модуля",
"не вказано модель захисту під час використання декількох міток",
"Показати список відомих областей",
"Не вдалося отримати список мовних розширень модуля",
"Не вдалося завантажити модуль обробки",
"Назва групи для перетягування вкладки",
"Неочікуваний передчасний кінець потоку",
"Датчик кольорової ділянки з двома чипами",
"Вивести відомості щодо користувача і перевірити розпізнавання",
"Ідентифікатор повідомлення у каталозі",
"Перемістити обчислення незмінних у петлях за межі петель.",
"Не вдалося виконати читання з монітора",
"Мінімальна ширина кнопок в контейнері",
"параметри містив пароль з порожньою назвою",
"Буде встановлено при наступному вході",
"Попереджати про невіртуальні деструктори.",
"Показати коротке повідомлення щодо користування для команди",
"Апаратний блок призначення не збігається із джерелом",
"Ізольована мережа, лише внутрішня маршрутизація",
"Висота відеозапису, одержаного з камери, в пікселях",
"Процесором основної системи не надаються потрібні можливості",
"Немає повідомлення бажаного типу",
"Чи інтеграцію з оболонкою ввімкнено",
"Показувати під час запуску стовпчик використання процесору.",
"Не можна вилучати загальносистемні розширення",
"Команда автоматичної компенсації спалаху",
"Чи показується розмір вибраного шрифту у позначці",
"Не вдалося створити зображення попереднього перегляду друку.",
"кореневий сертифікат було позначено як надійний",
"Перейти до попереднього елемента списку",
"Чи показувати кнопку закривання на панелі інструментів",
"Базовий рельєф з двома джерелами світла",
"недійсні типи у конвертації до цілого числа",
"Не вдалося виконати пошук файла",
"У назві домену не повинно міститися символів розриву рядка",
"регістр джерела збігається з основою зворотного запису",
"не вдалося завантажити дані символів",
"Експортувати обрізаний вміст у сторінці",
"читання користувацьких методів доступу",
"Масштабувати за центром сторінки",
"перевизначити типове розташування кореневої теки системи",
"обʼєкт призначення виділяється тут",
"Визначає відстань до точки призначення.",
"Під час спроби вивантаження підшляху метаданих:",
"Для встановлення усіх наданих випусків потрібен пристрій",
"вивести дані у зручному для читання форматі",
"Не вдалося отримати адреси фрагмента.",
"елемент масиву позначок не є рядком",
"Внутрішня помилка: невідома помилка",
"Автономний регіон Азорські острови",
"Встановити базовою одиницею виміру сантиметр",
"Рядок, що показано в позначці вкладки дочірнього елемента",
"Визначає спосіб малювання тіні навколо порту перегляду",
"помилка позиціонування голівки сканування",
"встановити ширину каналу перенесення після копіювання",
"Керування динамічними правами доступу",
"Друкувати виконані операції і код завершення команди",
"Неможливо започаткувати аналізатор файлів конфігурації.",
"Показати додаткові діагностичні дані",
"Прочитати кутку зі стандартного джерела вхідних даних",
"визначити батьківський запис поточного знімка",
"Не можна мати розділи, які перекриваються.",
"ігноруємо некоректний необроблений пароль",
"Неможливо перейменувати файли верхнього рівня",
"конфліктуючі рядки заміни для порожнього поля",
"Глянцевий фотопапір найвищої якості",
"мало бути вказано регістр цілих чисел",
"Клавіатурне скорочення для очищення підсвічування знайденого",
"Використовувати лише апаратні значення",
"В цьому файлі немає динамічного розділу.",
"Виберіть вузол для отримання форми входу",
"відповідного пристрою файлової системи не знайдено",
"Файл сертифіката для перевірки сервера",
"показати контрольні суми повідомлень",
"Не вдалося відкрити пристрій тимчасового сховища ключів.",
"Отримати криві з позначеного для заміни поточного гліфу",
"небезпечний непрямий виклик функції в межах атомної транзакції",
"використання застарілих параметрів налаштовування",
"і кожен тип може бути перетворений на інший",
"Символи запису музики знаменного співу",
"сміття після числового літерала",
"доступ до тимчасових індексів з інших сесій заблокований",
"Не вдалося прочитати директорію завантаження",
"Припустити, що переповнення вказівника обгортається.",
"Проведіть вашим пальцем вздовж пристрою для зчитування",
"Демонтувати пристрій, змонтований іншим користувачем",
"сертифікат не мав використовуватися для підписування",
"для зміни типу потрібно бути суперкористувачем",
"Визначає, чи слід вмикати пришвидшення обробки плоских полотен",
"Горно-Бадахшанська автономна область",
"Автоматичне злиття не спрацювало.",
"недостатньо даних експорту для читання",
"Завершення роботи за бажанням користувача",
"Створити проміжки у самоперетинах, як у кельтських вузлах",
"неможливо додати контракти до віртуальної функції",
"Для виконання дій з файлами слід пройти розпізнавання",
"сертифікат непридатний для підписування",
"Увімкнути евристику підрахунку залежностей в планувальнику.",
"Підтримки перемикання розділів у коді не передбачено.",
"Нова подія наступного понеділка",
"Взяти видимий колір і прозорість",
"Не вдається задовольнити всі обмеження на розділ.",
"подвійні константи не підтримуються",
"Накопичувати зсув для кожного стовпчика",
"Пересунути спосіб введення нижче",
"оновити, навіть якщо індекс містить не злиті записи",
"Назва файла залежностей, який слід створити",
"Перемістити плоский перегляд на кінець рядка",
"помилка під час виконання швидкого експорту",
"не вдалося отримати дані щодо типу долучення носія даних",
"не вдалося додати подію до черги обробки",
"Проведення ліворуч двома пальцями",
"Сертифікат не містить відкритого ключа",
"Перед використанням пристрій не було зарезервовано",
"елементи керування відтворенням та показом стану аудіо",
"типи не можуть бути визначені в умовах",
"некоректний заголовок прозорого підпису",
"Перелік символічних імен і еквівалентів кольорів.",
"Показати діаграму процесора як багатоярусну діаграму",
"Виберіть зірку, щоб залишити оцінку",
"Очікуємо на носій або джерело пакунків для встановлення",
"під час спроби визначити розмір файлової системи",
"Розмір кроку гучності для кожної зміни гучності",
"У автоматичному режимі явні зупинки ігноруються",
"Не вдалося обробити список сеансів",
"Стан апаратного захисту та відомості щодо нього",
"зовнішня команда для перевірки віртуальних оболонок",
"пропущено вираз для ширини паперу",
"Вимикання системи, коли інші користувачі ще у ній",
"Позначте, щоб зробити шрифт курсивним.",
"неприпустиме використання атрибутів у порожньому оголошенні",
"Довжина циклу блимання курсора, в мілісекундах",
"Завантажити систему у режимі єдиного користувача.",
"Надавачі даних щодо обмінних курсів",
"Операція встановлення позиції не підтримується для потоків",
"таку назву інтерфейсу зарезервовано",
"потрібні деякі комміти для відтворення",
"жодного псевдоніму для стовпця не було надано",
"Перетягніть, щоб змінити позицію у стосі ефектів контуру",
"Для вставлення розширення до вузла не вистачає місця",
"Іріан Джайя та Молуккські острови",
"Розпочато роботу з повторного трасування",
"Незбережені зміни буде втрачено без можливості відновлення.",
"Програмі не вдалося знайти жодного доступного пакунка.",
"Не вдалося встановити сталість інтерфейсу тунелю",
"Обмеження на вільний дисковий простір",
"Реєстраційні дані для доступу до реєстру джерела",
"Віджет піктограми для показу в пункті",
"не вдалося знайти звуковий модуль для звукового пристрою",
"Не вдалося перевірити кореневий хеш.",
"Одиниця вимірювання для відстані до цілі",
"Вибрати колір для варіантів з бази даних користувача",
"присвоєння виразу з типом масиву",
"показати дані щодо проблем із наданням залежностей",
"для параметра функції масиву необхідно вказати вираз довжини",
"Щоб відкрити пристрій, слід пройти розпізнавання",
"Скористатися файлом як вхідним списком дій",
"Не вдалося отримати кількість контрольних точок",
"Немає надзвичайної дії з монтування",
"Екран, на якому буде виведено це вікно",
"Попереджати, якщо простір адреси змінюється.",
"не запускати і не зупиняти служби",
"Не вдалося визначити час, коли цю дію було виконано востаннє",
"Порівняти версії, які вказано як аргументи.",
"НАЗВА РОЗТАШУВАННЯ - Додати віддалене сховище",
"засновано на вашому паролі для входу",
"Розмір значків у цій панелі інструментів",
"перевизначити метадані для поточного знімка",
"Ви ввели правильний пароль. Спробуйте знову.",
"Для завершення дії на диску недостатньо вільного місця",
"Виключити типові каталоги зі шляху пошуку файлів",
"Збирати інформацію про команди які виконуються.",
"Вивести діагностичну інформацію про оптимізацію.",
"Клацніть, щоб відшукати символ.",
"Додавання нового віртуального обладнання",
"Не забезпечено методів розпізнавання",
"Документація, яка може допомогти:",
"вказано базовий регістр, але нульовий",
"Пошук і заміна у межах документа",
"некоректний результат оптимізації фрагмента",
"Показувати розділи для обробки виключень",
"Вставлено новий рядок або стовпчик.",
"неможливо створити тимчасове відношення в не тимчасовій схемі",
"Вийти з оболонки і повернутися до головного меню",
"Неможливо скопіювати файл сам у себе.",
"Помилковий вираз поточного значення",
"Чи цей рядок заголовку слід сховати, коли вікно розгорнуто",
"Стиль підкреслення цього тексту",
"немає завдання із резервного копіювання домену",
"Помилка при виводі декодованого шаблону",
"Встановлює кількість часу для оновлення файлу журналу.",
"Не вдалося отримати дані щодо геометрії диска.",
"Розмір вектора не є цілочисельним кратним розміру компоненти",
"Не вдалося оновити мікропрограму:",
"Можливі значення параметра СТИЛЬ:",
"Інформація про властивості пристрою:",
"ціль не є вказівником або посиланням на клас",
"Визначити значення змінної за введеними користувачем даними.",
"вилучити пакунок або пакунки з вашої системи",
"Не вистачає властивості ідентифікатора ОС розгортання",
"Вказати процесор для моделі конвеєра.",
"файл з записами щодо даних користувачів",
"Ширина, в точках, ліній рівня вкладення та ліній сітки",
"Більше немає відвіданих посилань.",
"не є числом, використовуємо нуль.",
"Перегляд і налаштовування скорочень",
"Радіус вікна, яке буде проаналізовано",
"не дозволяється безіменне перелічування з областю видимості",
"Віджет, який зараз показано у стосі",
"Наступний вузол за порядком читання вузлів",
"Не записувати дані до бази даних журналу",
"потік перервано іншим потоком обробки",
"Несподівана бітова глибина для елементів мапи кольорів",
"Використовувати нетиповий шрифт для мовної панелі",
"імпортувати визначення таблиць зі стороннього серверу",
"обчислене значення не використовується",
"Вказано невідомий алгоритм або протокол.",
"Носій не має ідентифікатора, неможливо вилучити",
"не вдалося визначити символ екранування",
"Некоректний операнд: поточне значення використано як адресу.",
"кількість процесорів є надто великою",
"Початкова точка для визначення початкового кута",
"за допомогою зовнішнього скрипту компонування:",
"Не вибрано жодного джерела введення",
"Визначає, чи є видимим перемикач, який вмикає розгортання",
"Перейти до робочого простору праворуч",
"Модуль зараз виконує демонтування",
"Прилипання лише до вузла, найближчого до вказівника",
"Чи повинні розкривні елементи мати лінію відриву",
"Увімкнути або вимкнути екранну клавіатуру",
"не вказано архітектурного розширення",
"не вдалося створити вихідні файли",
"Ширина стовпчика сеансу процесу",
"для цього буфера сховища даних слід вказати місткість тому",
"Тип переспрямовування служби і мережі",
"Придатні до встановлення атрибути:",
"Успішно вимкнено запис віддаленого сховища",
"Колір переднього плану у вигляді рядка",
"Вказує які сповіщення показуються і що вони показують",
"Ви знайдете свій файл у каталозі Звантаження.",
"Не вдалося передати дані в вікно",
"назви можливостей, відокремлені комами",
"Створити чотири напрямні за краями поточної сторінки",
"не виводити список згорнутих ідентичних розділів",
"Не вдалося побудувати контекст обробки",
"Компілювати код для режиму великого порядку байтів.",
"Показати користувача, яким було внесено зміну",
"рядок починається або завершується на заборонений дефіс",
"отримати блокову статистику пристроїв для домену",
"Помилка введення-виведення під час розшифрування слоту ключів.",
"неможливо порівняти іменований канал з директорією",
"Немає відомостей щодо насильства у мультфільмах",
"Наразі ви редагуєте коміт при перебазуванні.",
"Можете залишити панельні вікна тут.",
"занадто багато значень у вказівці повернення",
"Будь ласка, введіть відомості щодо вашого вторинного ключа:",
"Вікно заблоковано. Натисніть, щоб унести зміни",
"Передчасний кінець регулярного виразу",
"Чи дозволяти зміну постачальника встановлених залежностей.",
"Помилка читання зображення з карти",
"Показати інформацію щодо вказаного файла",
"Підрозділів на основну кругову поділку:",
"не можна поєднувати пре- і постіндексування",
"Помилка під час спроби обробки даних вводу-виводу агента",
"Не вдалося прочитати вхідні дані користувача",
"глобальна кваліфікація імені класу недійсна",
"Скоригувати точку дотику дотичної",
"Довжина проміжку між стібками при показі стібків",
"неможливо підключити предка успадкування в якості секції",
"виконати команду у фоновому режимі",
"Комбінація клавіш для відкривання нової вкладки",
"невідома помилка запису в стандартний вивід",
"Чи слід показувати лінії рівня вкладення у віджеті",
"Щоб звантажити альбом, потрібно вказати адресу фонотеки.",
"Приведення дерева до початкового стану...",
"неприпустимий шлях до віддаленої служби",
"Рядок, який буде використано для некоректних символів",
"ваша поточна гілка виглядає пошкодженою",
"Позиція у символах, на якій слід показувати праве поле.",
"Закриті частини основного ключа зберігаються на картці.",
"Апаратна або програмна рухома крапка",
"Створити розділ на нерозподіленому просторі",
"додати більше причин тайм-ауту не можна",
"Датчик ділянки послідовності кольорів",
"Налаштовування мобільного широкосмугового пристрою",
"Помилка служби виявлення пристроїв.",
"Вимовляти координати комірок таблиці",
"Повернена помилка з порожнім тілом",
"не вдалося записати граф комітів",
"Низький заряд батареї гучномовця",
"мало бути використано попередньо індексований вираз",
"сертифікат має ПОМИЛКОВИЙ підпис",
"Додавання опорної точки градієнта",
"Розділювач полів - нульовий байт.",
"Загальний час завантаження у пристрій",
"доступ до тимчасових таблиць з інших сесій заблоковано",
"Не вдалося увійти до жодного з виявлених вузлів",
"Схоже, ви виміряли не ту смугу.",
"Помилка експортування растрових даних",
"не вдалося створити пару сокетів",
"Спроба виконання дії була невдалою",
"Перейти вниз на наступний рядок",
"Не вдалося створити ідентифікатор наступного класу",
"Генерувати інструкцію повернення в голій функції.",
"необхідний шаблон текстового пошуку",
"Не вдалося отримати список буферів",
"заборонене використання керівного регістра",
"Вихідний статус визначається ВИРАЗОМ.",
"Подробиці: проксі не було створено.",
"Пошук потрібних пакунків у сховищах",
"Номер акумулятора виходить за межі",
"некоректне значення кількості активних процесорів вузла",
"Стара версія програмного інтерфейсу",
"Пропустити поточне вікно під час пошуку",
"останній аргумент повинен бути негайним значенням",
"Вибирати тему кнопок зі стрілками у вікні варіантів",
"профіль петлі не може бути портом",
"виявлено помилкове кодування адреси",
"Немає придатного до використання жетона.",
"Подвоїти оптичну роздільну здатність",
"подальші попередження щодо багатобайтових символів придушено",
"перегляд неактивних та активних доменів",
"Показує версію сервера у вигляді цілого числа.",
"Перейти на рівень вгору ієрархією документа",
"суперечливі параметри визначення ширини",
"Показувати тимчасовий обрис контуру",
"непідтримуваний розмір змінної або значення заповнення",
"Отримати список всіх доступний профілів кольорів",
"Підтримка користування на малому екрані",
"Збереження документа як шаблону",
"Перейти до попереднього перехресного посилання",
"Не використовуйте апаратне з плаваючою комою.",
"регістр індексу перериває регістр перенесення",
"Неможливо створити директорію кешу метаданих.",
"Долучення до цієї області неможливе",
"Дата останнього доступу до файлу користувачем.",
"Запросити у користувача, якщо потрібна перевірка автентичності",
"не можна одночасно знищити і функцію і змінну",
"Є повний опис терміналу. Всі клавіші працюють.",
"Випадкова варіація довжини ліній побудови",
"зберігати коміти, які починаються порожніми",
"гігабайт,гігабайти,гігабайтів,ГБ",
"Наблизити ефект викликів функцій для спрощення аналізу.",
"Акумулятор не є сталим цілочисельним",
"Не вдалося експортувати перевизначення користувача",
"Перемкнутися на режим півширинних літер",
"другий, третій і четвертий аргументи повинні бути константами",
"показати список лише активних буферів",
"У потоці міститься недостатньо даних.",
"максимальний розмір кожного файла пакунка",
"Функція оптично-електронного перетворення",
"Мітка для пропозицій від засобу перевірки правопису",
"Показувати параметри вибору файла"
]

def load_system_corpus():
    """Читаємо всі .mo-файли української локалі. Повертаємо трійки
    (програма, англійський оригінал, український переклад)."""
    docs = []
    for path in sorted(glob.glob('/usr/share/locale/uk/LC_MESSAGES/*.mo')):
        try:
            with open(path, 'rb') as f:
                catalog = gettext.GNUTranslations(f)
        except Exception:
            continue                    # зламаний або чужий формат — просто пропускаємо
        program = path.split('/')[-1][:-3]
        for source, target in catalog._catalog.items():
            # службовий заголовок каталогу має ключ '' і не є текстом
            if isinstance(source, str) and isinstance(target, str) \
               and len(target) > 30 and 'Project-Id' not in target:
                docs.append((program, source, target))
    return docs


corpus = load_system_corpus()
if len(corpus) < 5000:
    corpus = [('fallback', '', text) for text in FALLBACK]
    print("⚠️  системної локалі немає, працюємо на вбудованому корпусі")
else:
    print("шлях спрацював: /usr/share/locale/uk/LC_MESSAGES/*.mo")

# розміри вибірок: великий — для головного заміру, решта врізані заради часу
N_HEAD, N_LAB, N_TOPICS, N_TASK = 20000, 1200, 800, 3000
if len(corpus) < 5000:
    N_HEAD, N_LAB, N_TOPICS, N_TASK = 400, 300, 250, 300

print("документів:", len(corpus))
print("розміри вибірок:", N_HEAD, N_LAB, N_TOPICS, N_TASK)
print()
for program, source, target in corpus[:3]:
    print(f"[{program}] {target[:70]!r}")

## 3 · Токенізатор і стоп-список із частот

Токенізатор — спільний для всього блоку. Він трактує апостроф як звʼязку
**всередині** слова, тож «зʼєднання» лишається одним токеном:

    TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"

Готового доброго списку стоп-слів для української немає, тому будуємо його самі:
беремо **сорок** найчастіших словоформ усього корпусу. Той самий прийом уживала
тема 03.

In [ ]:
TOKEN_PATTERN = r"[а-яїієґ]+(?:['ʼ’][а-яїієґ]+)*"
tokenize = re.compile(TOKEN_PATTERN).findall

# частоти рахуємо по ВСЬОМУ корпусу, а не по вибірці — щоб список не плавав
word_counts = Counter()
for _, _, target in corpus:
    word_counts.update(tokenize(target.lower()))

STOP_40 = [word for word, _ in word_counts.most_common(40)]

print("слововживань:", sum(word_counts.values()), " словоформ:", len(word_counts))
print()
print("топ-40 — це наш стоп-список:")
print(" ".join(STOP_40))


def sample_texts(how_many, seed):
    """Випадкова вибірка документів. Зерно міняє САМЕ ВИБІРКУ — це важливо
    для розділу про стійкість, де ми окремо міняємо зерно моделі."""
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(corpus))[:min(how_many, len(corpus))]
    return [corpus[i][2] for i in order]

## 4 · Головний замір: LDA і NMF на сирому мішку слів

20 000 документів, `min_df=5` (слово має бути хоч у пʼяти документах),
`max_df=0.5` (і не більш ніж у половині). Це налаштування «за замовчуванням
розумно», з якими зазвичай і починають.

Просимо вісім тем в обох моделей і засікаємо час — **двома годинниками одразу**.
`time.time()` каже, скільки ти чекав; `time.process_time()` — скільки роботи
процесор справді зробив. На машині, зайнятій чимось іще, перше число роздувається,
а друге ні. Цитувати треба друге.

In [ ]:
K = 8

texts_head = sample_texts(N_HEAD, 0)
vectorizer = CountVectorizer(min_df=5, max_df=0.5, token_pattern=TOKEN_PATTERN)
X_head = vectorizer.fit_transform(texts_head)
words_head = vectorizer.get_feature_names_out()

filled = 100 * X_head.nnz / (X_head.shape[0] * X_head.shape[1])
print("матриця:", X_head.shape, " ненульових:", X_head.nnz, f" заповнено {filled:.4f} %")


def time_of(make_model, repeats=1):
    """Час навчання двома годинниками. Стінний (time.time) міряє, скільки ти
    чекав; процесорний (time.process_time) — скільки роботи справді зроблено.
    На спільній машині перший роздувається чужими задачами, другий — ні."""
    wall, cpu = [], []
    for _ in range(repeats):
        start_wall, start_cpu = time.time(), time.process_time()
        model = make_model()
        wall.append(time.time() - start_wall)
        cpu.append(time.process_time() - start_cpu)
    return model, float(np.mean(wall)), float(np.mean(cpu))


lda_head, WALL_LDA, TIME_LDA = time_of(
    lambda: LatentDirichletAllocation(n_components=K, random_state=0).fit(X_head))
nmf_head, WALL_NMF, TIME_NMF = time_of(
    lambda: NMF(n_components=K, random_state=0).fit(X_head), repeats=5)

print(f"LDA: процесорний {TIME_LDA:.1f} c · стінний {WALL_LDA:.1f} c, ітерацій {lda_head.n_iter_}")
print(f"NMF: процесорний {TIME_NMF:.3f} c · стінний {WALL_NMF:.3f} c, ітерацій {nmf_head.n_iter_}"
      f" (середнє з пʼяти прогонів)")
print(f"LDA повільніший у {TIME_LDA / TIME_NMF:.1f} раза за процесорним часом")
print(f"                у {WALL_LDA / WALL_NMF:.1f} раза за стінним")
print()
print("Стінний час залежить від того, чим машина зайнята ще; процесорний — ні.")
print("Далі всюди цитуємо процесорний.")

Тепер найголовніше — подивимось на самі теми. `components_` — це матриця
«тема × слово»; беремо з кожного рядка десять найбільших ваг.

In [ ]:
def top_words(model, vocabulary, how_many=10):
    """Десять слів із найбільшою вагою в кожній темі — те єдине, що людина
    насправді читає, дивлячись на тематичну модель."""
    result = []
    for topic_weights in model.components_:
        order = topic_weights.argsort()[::-1][:how_many]
        result.append([vocabulary[i] for i in order])
    return result


top_lda_head = top_words(lda_head, words_head)
top_nmf_head = top_words(nmf_head, words_head)

for name, topics in (("LDA", top_lda_head), ("NMF", top_nmf_head)):
    print(name)
    for number, topic in enumerate(topics):
        print(f"  тема {number}: " + " ".join(topic))
    print()

## 5 · Міра читабельності

«Стало краще» — не замір. Потрібне число, і воно не має залежати від того, що
саме ми прибираємо, — інакше міра стане круговою.

Беремо **закриті класи** української: прийменник, сполучник, частка, займенник.
Це списки, які не поповнюються, тож вони не залежать від нашого корпусу.
`pymorphy3` уміє їх називати. Слово вважаємо службовим, якщо **хоч один** його
розбір потрапляє в закритий клас — не «перший», бо перший часто хибний.

Девʼятнадцять слів довелося додати руками: звʼязку «є», модальні «може», «слід»,
«має» морфологія вважає дієсловами, а змісту вони не несуть. Це чесна вада міри.

In [ ]:
morph = pymorphy3.MorphAnalyzer(lang='uk')

CLOSED_POS = {'PREP', 'CONJ', 'PRCL', 'NPRO'}      # прийм., сполучн., частка, займенн.
EXTRA_FUNCTION = {'є', 'бути', 'буде', 'будуть', 'був', 'була', 'було', 'були',
                  'може', 'можуть', 'можна', 'має', 'мають', 'слід', 'треба',
                  'ще', 'вже', 'дуже', 'також', 'тільки'}

_function_cache = {}

def is_function_word(word):
    """Службове слово — те, що належить закритому класу мови.
    Перевіряємо ВСІ розбори: для «під» перший розбір — іменник (черінь печі),
    і лише другий — прийменник."""
    if word not in _function_cache:
        _function_cache[word] = (word in EXTRA_FUNCTION or
                                 any(p.tag.POS in CLOSED_POS for p in morph.parse(word)))
    return _function_cache[word]


print("перевірка міри на очевидних словах:")
for word in ['не', 'для', 'у', 'щодо', 'цей', 'є', 'можна',
             'файл', 'помилка', 'пристрій', 'час', 'вдалося']:
    print(f"   {word:<12} {'службове' if is_function_word(word) else 'змістовне'}")

Дві міри разом. Перша — частка службових слів у топ-10 усіх тем. Друга —
різноманіття: скільки **різних** слів у зведеному топ-10. Погані теми повторюють
одні й ті самі прийменники, тож різноманіття падає разом із якістю.

In [ ]:
def function_share(topics):
    """Частка службових слів серед усіх слів топ-10 усіх тем."""
    total = sum(len(topic) for topic in topics)
    hits = sum(is_function_word(word) for topic in topics for word in topic)
    return hits / total


def diversity(topics):
    """Скільки різних слів у зведеному топ-10: 1.0 — жодного повтору."""
    total = sum(len(topic) for topic in topics)
    return len({word for topic in topics for word in topic}) / total


print(f"LDA  служб@10 {function_share(top_lda_head):.4f}   "
      f"різноманіття {diversity(top_lda_head):.4f}   час {TIME_LDA:.1f} c")
print(f"NMF  служб@10 {function_share(top_nmf_head):.4f}   "
      f"різноманіття {diversity(top_nmf_head):.4f}   час {TIME_NMF:.3f} c")
print()
print("більше половини слів у темах LDA — прийменники й частки")

## 6 · Лематизація

`pymorphy3` зводить «файла», «файлу», «файли» до «файл». Тема 03 показувала,
що для української це стискає словник майже вдвічі — і водночас іноді помиляється.
Обидві властивості нам далі знадобляться.

In [ ]:
_lemma_cache = {}

def lemma(word):
    """Словникова форма. Кешуємо: слів у корпусі багато, а різних форм мало."""
    if word not in _lemma_cache:
        _lemma_cache[word] = morph.parse(word)[0].normal_form
    return _lemma_cache[word]


def lemmatize(text):
    return " ".join(lemma(word) for word in tokenize(text.lower()))


STOP_40_LEMMAS = sorted({lemma(word) for word in STOP_40})

print("було :", "Не вдалося створити файли налаштувань")
print("стало:", lemmatize("Не вдалося створити файли налаштувань"))
print()
print("стоп-слів після зведення до лем:", len(STOP_40_LEMMAS))
print("приклад помилки лематизатора: «домену» ->", lemma("домену"))

## 7 · Драбина прибирання

Чотири кроки, кожен додається до попередніх. Три зерна на кожну точку — три
різні вибірки по 1 200 документів. Різниця, менша за розкид між зернами,
різницею не є.

In [ ]:
SEEDS = (0, 1, 2)
STEP_NAMES = ["сирий мішок слів", "+ стоп-слова (топ-40)", "+ max_df = 0.02",
              "+ лематизація", "+ TF-IDF замість частот"]


def build_matrix(step, seed, how_many=None):
    """Матриця «документ × слово» на заданому щаблі драбини."""
    texts = sample_texts(how_many or N_LAB, seed)
    if step >= 3:
        texts = [lemmatize(text) for text in texts]
        stop = STOP_40_LEMMAS
    else:
        stop = STOP_40
    options = dict(min_df=2, token_pattern=TOKEN_PATTERN,
                   max_df=0.02 if step >= 2 else 0.5)
    if step >= 1:
        options['stop_words'] = stop
    Vectorizer = TfidfVectorizer if step >= 4 else CountVectorizer
    vec = Vectorizer(**options)
    return vec.fit_transform(texts), vec.get_feature_names_out()


ladder = []
print(f"{'крок':<24}{'ознак':>7}{'LDA служб@10':>21}{'NMF служб@10':>21}{'різн LDA':>10}")
for step, step_name in enumerate(STEP_NAMES):
    shares_lda, shares_nmf, div_lda, div_nmf, n_features = [], [], [], [], []
    words_of_seed_0 = None
    for seed in SEEDS:
        X, vocabulary = build_matrix(step, seed)
        n_features.append(X.shape[1])
        lda = LatentDirichletAllocation(n_components=K, random_state=0).fit(X)
        nmf = NMF(n_components=K, random_state=0).fit(X)
        topics_lda, topics_nmf = top_words(lda, vocabulary), top_words(nmf, vocabulary)
        shares_lda.append(function_share(topics_lda)); div_lda.append(diversity(topics_lda))
        shares_nmf.append(function_share(topics_nmf)); div_nmf.append(diversity(topics_nmf))
        if seed == 0:
            words_of_seed_0 = (topics_lda, topics_nmf)
    ladder.append({'name': step_name, 'features': int(np.mean(n_features)),
                   'lda': shares_lda, 'nmf': shares_nmf,
                   'div_lda': div_lda, 'div_nmf': div_nmf,
                   'words': words_of_seed_0})
    print(f"{step_name:<24}{int(np.mean(n_features)):>7}"
          f"{np.mean(shares_lda):>14.4f}±{np.std(shares_lda):.4f}"
          f"{np.mean(shares_nmf):>14.4f}±{np.std(shares_nmf):.4f}"
          f"{np.mean(div_lda):>10.4f}")

Числа — це половина відповіді. Друга половина — самі слова: подивимось, як
міняється **перша тема** LDA на кожному щаблі.

In [ ]:
for step, row in enumerate(ladder):
    marks = "".join("*" if is_function_word(w) else "." for w in row['words'][0][0])
    print(f"{step} · {row['name']}")
    print("   " + " ".join(row['words'][0][0]))
    print(f"   службових у цій темі: {marks.count('*')} з 10")
    print()

## 8 · `max_df` замість списку

Драбина показала дивне: `max_df=0.02` **після** стоп-слів майже нічого не змінює.
Гіпотеза — це той самий важіль, узятий з іншого боку. Перевіримо: крутимо саме
`max_df` без жодного списку.

In [ ]:
print(f"{'max_df':>8}{'ознак':>8}{'служб@10':>18}{'різноманіття':>16}")
max_df_rows = []
for max_df in (0.005, 0.01, 0.02, 0.05, 0.1, 0.25, 0.5):
    shares, divs, features = [], [], []
    for seed in SEEDS:
        vec = CountVectorizer(min_df=2, max_df=max_df, token_pattern=TOKEN_PATTERN)
        X = vec.fit_transform(sample_texts(N_LAB, seed))
        topics = top_words(NMF(n_components=K, random_state=0).fit(X),
                           vec.get_feature_names_out())
        shares.append(function_share(topics)); divs.append(diversity(topics))
        features.append(X.shape[1])
    max_df_rows.append({'max_df': max_df, 'features': float(np.mean(features)),
                        'share': float(np.mean(shares)), 'sd': float(np.std(shares)),
                        'div': float(np.mean(divs))})
    print(f"{max_df:>8}{np.mean(features):>8.0f}"
          f"{np.mean(shares):>10.4f}±{np.std(shares):.4f}{np.mean(divs):>16.4f}")

print()
print(f"стоп-список із 40 слів давав на NMF: {np.mean(ladder[1]['nmf']):.4f}")
print("один поріг без жодного списку працює не гірше")

## 9 · NMF руками: розклад матриці

NMF шукає дві невідʼємні матриці `W` («документ × тема») і `H` («тема × слово»),
щоб їхній добуток був якомога ближчим до вихідної матриці `V`. Близькість
міряють нормою Фробеніуса: різниця матриць, кожна клітинка в квадрат, усе
додати, взяти корінь.

Перевіримо, що всередині `sklearn` немає магії: порахуємо цю норму самі.

In [ ]:
X_lab, words_lab = build_matrix(3, 0)          # щабель «лематизація», зерно 0

model = NMF(n_components=K, random_state=0)
W = model.fit_transform(X_lab)                 # документ × тема
H = model.components_                          # тема × слово

approximation = W @ H
by_hand = np.sqrt(((X_lab.toarray() - approximation) ** 2).sum())

print("наша похибка   :", f"{by_hand:.10f}")
print("sklearn        :", f"{model.reconstruction_err_:.10f}")
assert np.allclose(by_hand, model.reconstruction_err_), "розрахунок розійшовся!"
print("✅ збігається")
print()

cells_full = X_lab.shape[0] * X_lab.shape[1]
cells_decomposed = W.size + H.size
print(f"клітинок у матриці : {cells_full}")
print(f"чисел у розкладі   : {cells_decomposed}")
print(f"стиснення          : {cells_full / cells_decomposed:.1f}×")

І одна перевірка, яка коротше за абзац пояснює різницю двох моделей: у LDA рядок
теми — це справжні ймовірності слів, у NMF — просто ваги.

In [ ]:
lda_rows = lda_head.components_ / lda_head.components_.sum(axis=1, keepdims=True)
print("рядок LDA після нормування сумується до:", f"{lda_rows[0].sum():.10f}")
print("рядок NMF сумується до                 :", f"{H[0].sum():.4f}")
print()
print("саме тому LDA вміє сказати ймовірність нового документа, а NMF — ні")

## 10 · Скільки коштує час LDA

Різниця в часі — не дрібниця, і варто побачити, як вона росте. NMF на кожній
ітерації робить кілька множень матриць; LDA замість цього крутить окремий цикл
**на кожному документі**.

Три повтори на кожен розмір. Останній стовпчик — стінний час LDA: порівняй його
з процесорним і побачиш, наскільки машина зайнята чимось іще.

In [ ]:
print(f"{'документів':>11}{'LDA проц., c':>18}{'NMF проц., c':>20}{'відн.':>8}{'LDA стін.':>12}")
time_rows = []
for how_many in (500, 1000, 2000, 4000):
    cpu_lda, cpu_nmf, wall_lda = [], [], []
    for repeat in range(3):        # три повтори: одне вимірювання часу нічого не варте
        vec = CountVectorizer(min_df=5, max_df=0.5, token_pattern=TOKEN_PATTERN)
        X = vec.fit_transform(sample_texts(how_many, 0))
        _, wall, cpu = time_of(lambda: LatentDirichletAllocation(n_components=K, random_state=0).fit(X))
        cpu_lda.append(cpu); wall_lda.append(wall)
        _, _, cpu2 = time_of(lambda: NMF(n_components=K, random_state=0).fit(X))
        cpu_nmf.append(cpu2)
    time_rows.append({'n': how_many, 'lda': float(np.mean(cpu_lda)), 'nmf': float(np.mean(cpu_nmf))})
    print(f"{how_many:>11}{np.mean(cpu_lda):>12.2f}±{np.std(cpu_lda):.2f}"
          f"{np.mean(cpu_nmf):>13.3f}±{np.std(cpu_nmf):.3f}"
          f"{np.mean(cpu_lda)/np.mean(cpu_nmf):>8.1f}{np.mean(wall_lda):>12.2f}")

time_rows.append({'n': N_HEAD, 'lda': TIME_LDA, 'nmf': TIME_NMF})
print(f"{N_HEAD:>11}{TIME_LDA:>12.2f}{'':>6}{TIME_NMF:>13.3f}{'':>6}"
      f"{TIME_LDA/TIME_NMF:>8.1f}{WALL_LDA:>12.2f}")

## 11 · Скільки тем брати

Два кандидати на критерій. **Перплексія** LDA: наскільки модель здивована
документом, якого не бачила (менше — краще). **Помилка відновлення** NMF: та сама
норма Фробеніуса, поділена на норму матриці (менше — краще).

Обидві рахуємо на **відкладеній** чверті документів, три зерна.

In [ ]:
print(f"{'k':>3}{'перплексія LDA':>20}{'помилка NMF':>16}{'служб LDA':>12}{'служб NMF':>12}")
k_rows = []
for k in (2, 4, 8, 16):
    perplexities, errors, shares_lda, shares_nmf = [], [], [], []
    for seed in SEEDS:
        texts = [lemmatize(text) for text in sample_texts(N_TOPICS, seed)]
        vec = CountVectorizer(min_df=2, max_df=0.02, token_pattern=TOKEN_PATTERN,
                              stop_words=STOP_40_LEMMAS)
        X = vec.fit_transform(texts)
        vocabulary = vec.get_feature_names_out()
        train, test = train_test_split(np.arange(X.shape[0]), test_size=0.25,
                                       random_state=seed)

        lda = LatentDirichletAllocation(n_components=k, random_state=0).fit(X[train])
        perplexities.append(lda.perplexity(X[test]))

        nmf = NMF(n_components=k, random_state=0).fit(X[train])
        held_out = X[test].toarray()
        residual = held_out - nmf.transform(X[test]) @ nmf.components_
        errors.append(np.linalg.norm(residual) / np.linalg.norm(held_out))

        shares_lda.append(function_share(top_words(lda, vocabulary)))
        shares_nmf.append(function_share(top_words(nmf, vocabulary)))
    k_rows.append({'k': k, 'pp': float(np.mean(perplexities)), 'ppd': float(np.std(perplexities)),
                   'err': float(np.mean(errors)), 'errd': float(np.std(errors)),
                   'lda': float(np.mean(shares_lda)), 'nmf': float(np.mean(shares_nmf))})
    print(f"{k:>3}{np.mean(perplexities):>16.1f}±{np.std(perplexities):.0f}"
          f"{np.mean(errors):>16.4f}{np.mean(shares_lda):>12.4f}{np.mean(shares_nmf):>12.4f}")

print()
print("перплексія росте з k  -> найкраще k = 2")
print("помилка падає з k     -> найкраще найбільше k")
print("два критерії дивляться в протилежні боки, мінімуму немає ні в кого")

## 12 · Найкращі теми, які вдалося дістати

NMF на всіх 20 000 документах: топ-40 викинуто, лематизовано, `max_df=0.02`.
NMF тут не з ліні, а тому, що на цьому корпусі вона дає не гірші теми в десятки
разів швидше — і три зерна коштують секунди, а не хвилини.

In [ ]:
named_topics = []
for seed in SEEDS:
    texts = [lemmatize(text) for text in sample_texts(N_HEAD, seed)]
    vec = CountVectorizer(min_df=5, max_df=0.02, token_pattern=TOKEN_PATTERN,
                          stop_words=STOP_40_LEMMAS)
    X = vec.fit_transform(texts)
    nmf = NMF(n_components=K, random_state=0).fit(X)
    named_topics.append(top_words(nmf, vec.get_feature_names_out()))
    if seed == 0:
        print(f"ознак {X.shape[1]}, служб@10 {function_share(named_topics[0]):.4f}, "
              f"різноманіття {diversity(named_topics[0]):.4f}")
        print()

for number, topic in enumerate(named_topics[0]):
    print(f"тема {number}: " + " ".join(topic))

Назви цим спискам придумує **людина**, і не всім спискам вдається. Спробуй
назвати вісім тем сам, перш ніж читати розділ 09 лекції.

## 13 · Чи повторяться теми

Модель нумерує теми довільно, тож спершу треба **зіставити** їх у пари так, щоб
сумарна схожість була найбільшою. Це класична задача про призначення; у `scipy`
для неї є готова функція. Схожість пари міряємо коефіцієнтом Жаккара.

І окремо: три зерна курсу міняють **дані**, а не **прогін**. Тому міряємо обидва
джерела нестійкості нарізно — і додаємо рівень випадковості, без якого числа
нічого не означають.

In [ ]:
def jaccard(first, second):
    """Скільки слів спільні, поділити на скільки слів разом."""
    a, b = set(first), set(second)
    return len(a & b) / len(a | b)


def matched_similarity(topics_a, topics_b):
    """Зіставляємо теми в пари найвигіднішим чином і повертаємо середню схожість."""
    table = np.array([[jaccard(a, b) for b in topics_b] for a in topics_a])
    rows, columns = linear_sum_assignment(-table)      # мінус, бо шукаємо максимум
    return float(table[rows, columns].mean()), table


X_stab, words_stab = build_matrix(3, 0)

# 1) ті самі дані, різний random_state — нестійкість САМОЇ МОДЕЛІ
same_data = [top_words(LatentDirichletAllocation(n_components=K, random_state=state).fit(X_stab),
                       words_stab) for state in (0, 1, 2)]
pairs_state = [matched_similarity(same_data[a], same_data[b])[0] for a, b in ((0,1),(0,2),(1,2))]

# 2) різні вибірки документів — нестійкість від ДАНИХ
per_seed = [build_matrix(3, seed) for seed in SEEDS]
other_data = [top_words(LatentDirichletAllocation(n_components=K, random_state=0).fit(X), voc)
              for X, voc in per_seed]
pairs_data = [matched_similarity(other_data[a], other_data[b])[0] for a, b in ((0,1),(0,2),(1,2))]

# 3) NMF на 20 000 документах, різні вибірки
pairs_nmf = [matched_similarity(named_topics[a], named_topics[b])[0] for a, b in ((0,1),(0,2),(1,2))]

# 4) рівень нуля: два випадкові набори по десять слів із того самого словника
rng = np.random.default_rng(0)
chance = []
for _ in range(20):
    fake_a = [list(rng.choice(words_stab, 10, replace=False)) for _ in range(K)]
    fake_b = [list(rng.choice(words_stab, 10, replace=False)) for _ in range(K)]
    chance.append(matched_similarity(fake_a, fake_b)[0])


def shared_words(jaccard_value):
    """Із Жаккара назад у «скільки слів із десяти спільні»: j = x / (20 - x)."""
    return 20 * jaccard_value / (1 + jaccard_value)


for label, values in (("лише random_state, дані ті самі", pairs_state),
                      ("інша вибірка документів       ", pairs_data),
                      ("NMF на 20 000, інша вибірка   ", pairs_nmf),
                      ("випадкові списки (рівень нуля)", chance)):
    print(f"{label}  Жаккар {np.mean(values):.4f}±{np.std(values):.4f}"
          f"   спільних слів із 10: {shared_words(np.mean(values)):.1f}")

print()
print("── зіставлені пари тем: ті самі дані, random_state 0 проти 1 ──")
_, table = matched_similarity(same_data[0], same_data[1])
rows, columns = linear_sum_assignment(-table)
for a, b in zip(rows, columns):
    common = sorted(set(same_data[0][a]) & set(same_data[1][b]))
    print(f"  тема {a} ↔ тема {b}: спільних {len(common)} — {' '.join(common) if common else '—'}")

### Стабільність окремо від відтворюваності

Попередній замір міняв разом і документи, і словник — інша вибірка дає інший
словник, і теми просто **не мають чим** збігтися. Чесніший дослід залишає простір
ознак незмінним: беремо один пул із 1 200 документів, будуємо словник **на всьому
пулі**, а далі вчимо моделі на трьох різних **підвибірках по 80 %** цього пулу.

Тепер міняються тільки документи. Якщо теми — це «прихована структура корпусу»,
вони мусять пережити викидання пʼятої частини документів.

In [ ]:
# словник будуємо на ВСЬОМУ пулі, щоб підвибірки жили в одному просторі ознак
pool_texts = [lemmatize(text) for text in sample_texts(N_LAB, 0)]
pool_vec = CountVectorizer(min_df=2, max_df=0.02, token_pattern=TOKEN_PATTERN,
                           stop_words=STOP_40_LEMMAS)
X_pool = pool_vec.fit_transform(pool_texts)
words_pool = pool_vec.get_feature_names_out()
print("пул:", X_pool.shape[0], "документів,", X_pool.shape[1], "ознак")

subsample_lda, subsample_nmf = [], []
for seed in SEEDS:
    rng = np.random.default_rng(100 + seed)
    keep = rng.permutation(X_pool.shape[0])[:int(0.8 * X_pool.shape[0])]
    part = X_pool[keep]
    subsample_lda.append(top_words(
        LatentDirichletAllocation(n_components=K, random_state=0).fit(part), words_pool))
    subsample_nmf.append(top_words(
        NMF(n_components=K, random_state=0).fit(part), words_pool))

pairs_sub_lda = [matched_similarity(subsample_lda[a], subsample_lda[b])[0]
                 for a, b in ((0, 1), (0, 2), (1, 2))]
pairs_sub_nmf = [matched_similarity(subsample_nmf[a], subsample_nmf[b])[0]
                 for a, b in ((0, 1), (0, 2), (1, 2))]

print(f"80 % пулу, LDA: Жаккар {np.mean(pairs_sub_lda):.4f}±{np.std(pairs_sub_lda):.4f}"
      f"   спільних слів із 10: {shared_words(np.mean(pairs_sub_lda)):.1f}")
print(f"80 % пулу, NMF: Жаккар {np.mean(pairs_sub_nmf):.4f}±{np.std(pairs_sub_nmf):.4f}"
      f"   спільних слів із 10: {shared_words(np.mean(pairs_sub_nmf)):.1f}")
print(f"рівень нуля     : Жаккар {np.mean(chance):.4f}")
print()
print("── пари тем LDA: підвибірка 1 проти підвибірки 2 ──")
_, table = matched_similarity(subsample_lda[0], subsample_lda[1])
rows, columns = linear_sum_assignment(-table)
for a, b in zip(rows, columns):
    common = sorted(set(subsample_lda[0][a]) & set(subsample_lda[1][b]))
    print(f"  тема {a} ↔ тема {b}: спільних {len(common)} — {' '.join(common) if common else '—'}")

## 14 · Грабля, якої три зерна не бачать

Зерно міняє дані. Якщо ж недетермінована **сама бібліотека**, три зерна цього не
викриють. Перевірка коротка й мусить стояти в кожній роботі, де є випадковість.

In [ ]:
first  = LatentDirichletAllocation(n_components=K, random_state=0).fit(X_stab).components_
second = LatentDirichletAllocation(n_components=K, random_state=0).fit(X_stab).components_
print("LDA, той самий стан і ті самі дані — побітово те саме:", np.array_equal(first, second))
assert np.array_equal(first, second), "LDA невідтворюваний!"

# NMF ініціалізується розкладом самої матриці (nndsvda), а не випадковими числами,
# тому random_state на неї майже не впливає
a = NMF(n_components=K, random_state=0).fit(X_stab).components_
b = NMF(n_components=K, random_state=7).fit(X_stab).components_
print("NMF, різні random_state — найбільша різниця ваги:", f"{np.abs(a - b).max():.2e}")

## 15 · Чи видно різницю LDA і NMF на справжній задачі

Задача з теми 06: відрізнити повідомлення про помилку від решти. Мітка береться
з **англійського** оригіналу, ознаки — з українського перекладу, тож модель не
бачить того, з чого зроблено мітку.

Замість слів даємо класифікаторові вісім чисел — розклад документа по темах.

In [ ]:
ERROR_WORDS = re.compile(r'\b(error|failed|cannot|could not|unable|invalid|denied|no such)\b',
                         re.I)

f1_lda, f1_nmf, f1_words = [], [], []
for seed in SEEDS:
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(corpus))[:min(N_TASK, len(corpus))]
    subset = [corpus[i] for i in order]
    labels = np.array([1 if ERROR_WORDS.search(source) else 0 for _, source, _ in subset])

    vec = CountVectorizer(min_df=2, max_df=0.02, token_pattern=TOKEN_PATTERN,
                          stop_words=STOP_40_LEMMAS)
    X = vec.fit_transform([lemmatize(target) for _, _, target in subset])

    lda = LatentDirichletAllocation(n_components=K, random_state=0).fit(X)
    nmf = NMF(n_components=K, random_state=0).fit(X)
    train, test = train_test_split(np.arange(X.shape[0]), test_size=0.3,
                                   random_state=seed, stratify=labels)
    for features, box in ((lda.transform(X), f1_lda),
                          (nmf.transform(X), f1_nmf),
                          (X, f1_words)):
        model = LogisticRegression(max_iter=1000, class_weight='balanced')
        model.fit(features[train], labels[train])
        box.append(f1_score(labels[test], model.predict(features[test])))

print("частка «помилка» у вибірці:", f"{labels.mean():.4f}")
print()
print(f"вісім тем LDA як ознаки : F1 {np.mean(f1_lda):.4f}±{np.std(f1_lda):.4f}")
print(f"вісім тем NMF як ознаки : F1 {np.mean(f1_nmf):.4f}±{np.std(f1_nmf):.4f}")
print(f"усі {X.shape[1]} слів як ознаки : F1 {np.mean(f1_words):.4f}±{np.std(f1_words):.4f}")
print()
print("різниця LDA і NMF:", f"{abs(np.mean(f1_lda)-np.mean(f1_nmf)):.4f}",
      "при розкидах", f"±{np.std(f1_lda):.4f}", f"±{np.std(f1_nmf):.4f}")
print("купи перетинаються — на цих даних різниці не видно")

## 16 · Числа, які цитує лекція

Одна клітинка, щоб можна було звірити лекцію із зошитом, не гортаючи вгору.

In [ ]:
print("── головний замір @", N_HEAD, "──")
print(f"матриця {X_head.shape[1]} ознак, {X_head.nnz} ненульових, {filled:.4f} % заповнено")
print(f"процесорний час: LDA {TIME_LDA:.1f} c · NMF {TIME_NMF:.3f} c · у {TIME_LDA/TIME_NMF:.1f} раза")
print(f"стінний час    : LDA {WALL_LDA:.1f} c · NMF {WALL_NMF:.3f} c · у {WALL_LDA/WALL_NMF:.1f} раза")
print(f"LDA служб@10 {function_share(top_lda_head):.4f} різноманіття {diversity(top_lda_head):.4f}")
print(f"NMF служб@10 {function_share(top_nmf_head):.4f} різноманіття {diversity(top_nmf_head):.4f}")
print()
print("── драбина @", N_LAB, "──")
for row in ladder:
    print(f"{row['name']:<24}{row['features']:>6} ознак  "
          f"LDA {np.mean(row['lda']):.4f}±{np.std(row['lda']):.4f} (різн {np.mean(row['div_lda']):.4f})  "
          f"NMF {np.mean(row['nmf']):.4f}±{np.std(row['nmf']):.4f} (різн {np.mean(row['div_nmf']):.4f})")
print()
print("── найкращі теми @", N_HEAD, "──")
print(f"служб@10 {function_share(named_topics[0]):.4f}  різноманіття {diversity(named_topics[0]):.4f}")
print()
print("── стійкість ──")
print(f"random_state {np.mean(pairs_state):.4f}±{np.std(pairs_state):.4f} · "
      f"дані {np.mean(pairs_data):.4f}±{np.std(pairs_data):.4f} · "
      f"NMF {np.mean(pairs_nmf):.4f}±{np.std(pairs_nmf):.4f} · "
      f"випадково {np.mean(chance):.4f}±{np.std(chance):.4f}")
print(f"80 % пулу: LDA {np.mean(pairs_sub_lda):.4f}±{np.std(pairs_sub_lda):.4f} · "
      f"NMF {np.mean(pairs_sub_nmf):.4f}±{np.std(pairs_sub_nmf):.4f}")
print()
print("── задача ──")
print(f"LDA {np.mean(f1_lda):.4f}±{np.std(f1_lda):.4f} · "
      f"NMF {np.mean(f1_nmf):.4f}±{np.std(f1_nmf):.4f} · "
      f"слова {np.mean(f1_words):.4f}±{np.std(f1_words):.4f}")

## Завдання трьох рівнів

**🟢 Рівень 1.** Візьми свій корпус коротких текстів (200-2000 документів) і
пройди по ньому драбину з розділу 7. Надрукуй `служб@10` на кожному щаблі.
**Зроблено, якщо** ти назвав щабель, який дав найбільший стрибок, і показав, що
цей стрибок більший за розкид по трьох зернах.

**🟡 Рівень 2.** Повтори розділ 11 для `k` від 2 до 30 і побудуй два графіки —
перплексію й помилку відновлення. **Зроблено, якщо** ти прямо написав, чи є на
котромусь із них лікоть, і якщо немає — сказав це словами, а не намалював його
собі.

**🔴 Рівень 3.** Заміряй стійкість інакше: замість Жаккара по топ-10 візьми
косинус між **повними** рядками `components_` зіставлених тем. **Зроблено,
якщо** ти назвав обидва числа для тих самих прогонів і пояснив, чому вони можуть
розходитись — і що з цього випливає для того, хто читає лише топ-10.

Повні умови — у [homework.html](homework.html).